# Heat Demand Analysis — EU Pulp & Paper Facilities (2024)

This notebook explores results from `calculate_heat_demand.py`:

- **Interactive map** of facilities colored by classification, sized by heat demand
- **Aggregations** by country and by product classification
- **Validation checks** against published production data and Eurostat sector benchmarks

Run `python3 heat_demand/calculate_heat_demand.py` first to refresh `heat_demand_by_facility.csv`.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Support running from project root or heat_demand/
_cwd = Path.cwd()
if (_cwd / "heat_demand" / "heat_demand_by_facility.csv").exists():
    PROJECT_ROOT = _cwd
elif (_cwd / "heat_demand_by_facility.csv").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd.parent

HEAT_DEMAND_PATH = PROJECT_ROOT / "heat_demand" / "heat_demand_by_facility.csv"
CLASSIFICATION_PATH = PROJECT_ROOT / "Input" / "pulp_paper_fossil_classification.xlsx"

MWH_TO_TJ = 0.0036  # 1 MWh_th = 3.6 GJ = 0.0036 TJ

CLASSIFICATION_COLORS = {
    "Pulp": "#1f77b4",
    "Integrated": "#ff7f0e",
    "Paper/Board": "#2ca02c",
    "Tissue": "#9467bd",
}

BAT_INTENSITY_RANGES_GJ_T = {
    "Pulp": (10, 16),
    "Integrated": (14, 20),
    "Paper/Board": (4, 6),
    "Tissue": (7, 12),
}

In [8]:
heat = pd.read_csv(HEAT_DEMAND_PATH)
meta = pd.read_excel(CLASSIFICATION_PATH, sheet_name="Facilities", header=1, engine="openpyxl")
fossil_shares = pd.read_excel(CLASSIFICATION_PATH, sheet_name="Fossil_Shares", header=1, engine="openpyxl")
eu_benchmark = pd.read_excel(CLASSIFICATION_PATH, sheet_name="EU_Benchmark_Verify", header=0, engine="openpyxl")

df = heat.merge(
    meta[[
        "source_id", "source_type", "justification", "annual_emissions_co2e_t",
        "total_TJ_country", "eurostat_country",
    ]],
    on="source_id",
    how="left",
)

df["heat_demand_tj"] = df["heat_demand_mwh_th"] * MWH_TO_TJ
df["replaceable_heat_tj"] = df["replaceable_heat_mwh_th"] * MWH_TO_TJ
df["implied_intensity_gj_per_t"] = (
    df["heat_demand_mwh_th"] * 3.6 / df["annual_output_t"]
)
df["has_coordinates"] = df["lat"].notna() & df["lon"].notna()

print(f"Facilities loaded: {len(df)}")
print(f"With coordinates: {df['has_coordinates'].sum()}")
print(f"With replaceable heat: {df['replaceable_heat_mwh_th'].notna().sum()}")
df.head()

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

## Portfolio summary

In [ ]:
summary = pd.DataFrame(
    {
        "Metric": [
            "Facilities",
            "Countries",
            "Total annual output (Mt)",
            "Total heat demand (TWh_th)",
            "Total replaceable heat (TWh_th)",
            "Avg fossil share (weighted)",
        ],
        "Value": [
            len(df),
            df["country_corrected"].nunique(),
            df["annual_output_t"].sum() / 1e6,
            df["heat_demand_mwh_th"].sum() / 1e6,
            df["replaceable_heat_mwh_th"].sum(skipna=True) / 1e6,
            df["replaceable_heat_mwh_th"].sum(skipna=True) / df["heat_demand_mwh_th"].sum(),
        ],
    }
)
summary

## Analysis by country

In [ ]:
by_country = (
    df.groupby("country_corrected", as_index=False)
    .agg(
        sites=("source_id", "count"),
        annual_output_mt=("annual_output_t", lambda s: s.sum() / 1e6),
        heat_demand_twh=("heat_demand_mwh_th", lambda s: s.sum() / 1e6),
        replaceable_heat_twh=("replaceable_heat_mwh_th", lambda s: s.sum(skipna=True) / 1e6),
        heat_demand_tj=("heat_demand_tj", "sum"),
        replaceable_heat_tj=("replaceable_heat_tj", lambda s: s.sum(skipna=True)),
        avg_fossil_share=("fossil_share_country", "mean"),
    )
    .sort_values("heat_demand_twh", ascending=False)
)
by_country["replaceable_share"] = (
    by_country["replaceable_heat_twh"] / by_country["heat_demand_twh"]
)
by_country

In [ ]:
fig_country = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Heat demand by country (TWh_th)", "Replaceable heat by country (TWh_th)"),
)
fig_country.add_trace(
    go.Bar(
        x=by_country["country_corrected"],
        y=by_country["heat_demand_twh"],
        marker_color="#4C78A8",
        name="Heat demand",
    ),
    row=1, col=1,
)
fig_country.add_trace(
    go.Bar(
        x=by_country["country_corrected"],
        y=by_country["replaceable_heat_twh"],
        marker_color="#F58518",
        name="Replaceable heat",
    ),
    row=1, col=2,
)
fig_country.update_layout(
    height=500, width=1100, showlegend=False,
    title="Country-level heat demand summary",
)
fig_country.update_xaxes(tickangle=-45)
fig_country.show()

## Analysis by classification

In [ ]:
by_class = (
    df.groupby("classification", as_index=False)
    .agg(
        sites=("source_id", "count"),
        annual_output_mt=("annual_output_t", lambda s: s.sum() / 1e6),
        heat_demand_twh=("heat_demand_mwh_th", lambda s: s.sum() / 1e6),
        replaceable_heat_twh=("replaceable_heat_mwh_th", lambda s: s.sum(skipna=True) / 1e6),
        avg_intensity_mwh_per_t=("heat_intensity_mwh_per_t", "first"),
        avg_fossil_share=("fossil_share_country", "mean"),
    )
    .sort_values("heat_demand_twh", ascending=False)
)
by_class

In [ ]:
fig_class = px.bar(
    by_class,
    x="classification",
    y=["heat_demand_twh", "replaceable_heat_twh"],
    barmode="group",
    color_discrete_sequence=["#4C78A8", "#F58518"],
    labels={
        "value": "TWh_th",
        "classification": "Product classification",
        "variable": "Metric",
    },
    title="Heat demand and replaceable heat by classification",
    category_orders={"classification": list(CLASSIFICATION_COLORS.keys())},
)
fig_class.update_layout(height=450, width=800)
fig_class.show()

In [ ]:
fig_pie = px.pie(
    by_class,
    names="classification",
    values="heat_demand_twh",
    color="classification",
    color_discrete_map=CLASSIFICATION_COLORS,
    title="Share of total heat demand by classification",
    hole=0.35,
)
fig_pie.update_layout(height=450, width=700)
fig_pie.show()

## Interactive map — heat demand per site

Marker **color** = product classification. Marker **size** = total heat demand (MWh_th/yr). Hover for site details.

In [ ]:
map_df = df[df["has_coordinates"]].copy()
map_df["marker_size"] = np.clip(
    map_df["heat_demand_mwh_th"] / map_df["heat_demand_mwh_th"].max() * 40 + 8,
    8, 48,
)

map_df["hover_text"] = (
    "<b>" + map_df["source_name"] + "</b><br>"
    + "Country: " + map_df["country_corrected"] + " (" + map_df["iso3_corrected"] + ")<br>"
    + "Classification: " + map_df["classification"] + "<br>"
    + "Confidence: " + map_df["confidence"].fillna("—") + "<br>"
    + "NACE: " + map_df["nace_used"] + "<br>"
    + "Annual output: " + map_df["annual_output_t"].map(lambda x: f"{x:,.0f} t/yr") + "<br>"
    + "Heat intensity: " + map_df["heat_intensity_mwh_per_t"].astype(str) + " MWh_th/t<br>"
    + "Heat demand: " + map_df["heat_demand_mwh_th"].map(lambda x: f"{x:,.0f} MWh_th/yr") + "<br>"
    + "Fossil share: " + map_df["fossil_share_country"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "N/A") + "<br>"
    + "Replaceable heat: " + map_df["replaceable_heat_mwh_th"].map(
        lambda x: f"{x:,.0f} MWh_th/yr" if pd.notna(x) else "N/A"
    )
)

fig_map = px.scatter_mapbox(
    map_df,
    lat="lat",
    lon="lon",
    color="classification",
    color_discrete_map=CLASSIFICATION_COLORS,
    size="marker_size",
    size_max=48,
    hover_name="source_name",
    custom_data=["hover_text"],
    zoom=3,
    center={"lat": 54, "lon": 12},
    mapbox_style="carto-positron",
    title="EU pulp & paper facilities — heat demand by site (2024)",
    category_orders={"classification": list(CLASSIFICATION_COLORS.keys())},
)
fig_map.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>",
)
fig_map.update_layout(height=700, width=1100, margin={"r": 0, "t": 50, "l": 0, "b": 0})
fig_map.show()

## Validation against published data

### 1. ENCE pulp mills (Spain) — production check

Reported 2020 production from [ENCE disclosures](https://ence.es/en/pulp-business/) vs Climate TRACE 2024 estimates.

In [ ]:
ence_published = pd.DataFrame(
    {
        "site": ["ENCE Navia", "ENCE Pontevedra"],
        "reported_production_t_yr": [572_567, 434_718],
        "reported_year": [2020, 2020],
        "source": "ENCE annual report / pulp business page",
    }
)

ence_estimated = df[df["source_name"].str.contains("ENCE", case=False, na=False)][[
    "source_name", "annual_output_t", "heat_demand_mwh_th", "classification",
    "heat_intensity_mwh_per_t", "fossil_share_country", "replaceable_heat_mwh_th",
]].copy()
ence_estimated["site"] = ence_estimated["source_name"].str.extract(
    r"(Navia|Pontevedra)", expand=False
)
ence_estimated["site"] = "ENCE " + ence_estimated["site"]

ence_validation = ence_published.merge(
    ence_estimated,
    on="site",
    how="left",
)
ence_validation["production_diff_pct"] = (
    (ence_validation["annual_output_t"] - ence_validation["reported_production_t_yr"])
    / ence_validation["reported_production_t_yr"] * 100
)
ence_validation[[
    "site", "reported_year", "reported_production_t_yr", "annual_output_t",
    "production_diff_pct", "heat_demand_mwh_th", "replaceable_heat_mwh_th",
]]

### 2. Heat intensity vs BAT reference ranges

Implied intensity = heat demand / annual output. Compare to BAT BREF ranges from the README (GJ/t).

In [ ]:
intensity_check = df[[
    "source_name", "country_corrected", "classification",
    "implied_intensity_gj_per_t", "heat_intensity_mwh_per_t",
]].copy()
intensity_check["bat_low_gj_per_t"] = intensity_check["classification"].map(
    lambda c: BAT_INTENSITY_RANGES_GJ_T[c][0]
)
intensity_check["bat_high_gj_per_t"] = intensity_check["classification"].map(
    lambda c: BAT_INTENSITY_RANGES_GJ_T[c][1]
)
intensity_check["within_bat_range"] = (
    (intensity_check["implied_intensity_gj_per_t"] >= intensity_check["bat_low_gj_per_t"])
    & (intensity_check["implied_intensity_gj_per_t"] <= intensity_check["bat_high_gj_per_t"])
)

print(
    "Facilities within BAT range:",
    intensity_check["within_bat_range"].sum(),
    f"/ {len(intensity_check)} (fixed SEC applied — expect all match chosen values)",
)
intensity_check.groupby("classification")["implied_intensity_gj_per_t"].agg(["mean", "min", "max"])

### 3. Country heat demand vs Eurostat sector energy (2023)

Compare summed facility heat demand (TJ) to Eurostat total final energy for the matching NACE sector.  
**Note:** Facility heat is modelled from production × SEC; Eurostat is reported final energy consumption — differences are expected but large gaps flag review.

In [ ]:
NACE_SECTOR_MAP = {
    "C1711": "C1711 — Pulp only",
    "C17xP": "C17_X_C1711 — Paper excl. pulp",
    "C17": "C17 — All paper & pulp",
}

facility_by_country_nace = (
    df.groupby(["eurostat_country", "nace_used"], as_index=False)
    .agg(
        sites=("source_id", "count"),
        modelled_heat_tj=("heat_demand_tj", "sum"),
        modelled_replaceable_tj=("replaceable_heat_tj", lambda s: s.sum(skipna=True)),
    )
)
facility_by_country_nace["sector"] = facility_by_country_nace["nace_used"].map(NACE_SECTOR_MAP)

eurostat = fossil_shares.rename(columns={
    "Country": "eurostat_country",
    "Total TJ (NCV)": "eurostat_total_tj",
    "Fossil Share": "eurostat_fossil_share",
})[["eurostat_country", "Sector", "eurostat_total_tj", "eurostat_fossil_share", "Data Available"]]

country_validation = facility_by_country_nace.merge(
    eurostat,
    left_on=["eurostat_country", "sector"],
    right_on=["eurostat_country", "Sector"],
    how="left",
).drop(columns=["Sector"])

country_validation["modelled_vs_eurostat_pct"] = (
    country_validation["modelled_heat_tj"] / country_validation["eurostat_total_tj"] * 100
)
country_validation = country_validation.sort_values("modelled_vs_eurostat_pct", ascending=False)
country_validation[[
    "eurostat_country", "nace_used", "sites", "modelled_heat_tj",
    "eurostat_total_tj", "modelled_vs_eurostat_pct", "eurostat_fossil_share", "Data Available",
]]

### 4. EU-level benchmark verification (from classification workbook)

In [ ]:
eu_benchmark_display = eu_benchmark.rename(columns={
    eu_benchmark.columns[0]: "metric",
    eu_benchmark.columns[1]: "eurostat_published",
    eu_benchmark.columns[2]: "derived_from_file",
    eu_benchmark.columns[3]: "match",
    eu_benchmark.columns[4]: "notes",
})
eu_benchmark_display = eu_benchmark_display.iloc[1:].reset_index(drop=True)
eu_benchmark_display

### 5. Data quality flags

Facilities flagged in the classification workbook for review (duplicates, mislabels, missing fossil share).

In [ ]:
df["status_flag"] = df["status_flag"].fillna("")
flagged = df[
    (df["status_flag"] != "")
    | df["fossil_share_country"].isna()
    | ~df["has_coordinates"]
].copy()

flag_summary = pd.DataFrame({
    "Issue": [
        "Any status flag",
        "Missing fossil share",
        "Missing coordinates",
        "Modelled heat > Eurostat country total (same NACE)",
    ],
    "Count": [
        (df["status_flag"] != "").sum(),
        df["fossil_share_country"].isna().sum(),
        (~df["has_coordinates"]).sum(),
        (country_validation["modelled_vs_eurostat_pct"] > 100).sum(),
    ],
})
flag_summary

In [ ]:
review_cols = [
    "source_name", "country_corrected", "classification", "status_flag",
    "fossil_share_country", "annual_output_t", "heat_demand_mwh_th",
    "replaceable_heat_mwh_th", "justification",
]
flagged[review_cols].sort_values(["status_flag", "country_corrected"])

## Export summary tables

In [ ]:
output_dir = PROJECT_ROOT / "heat_demand" / "analysis_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

by_country.to_csv(output_dir / "heat_demand_by_country.csv", index=False)
by_class.to_csv(output_dir / "heat_demand_by_classification.csv", index=False)
ence_validation.to_csv(output_dir / "validation_ence_production.csv", index=False)
country_validation.to_csv(output_dir / "validation_country_vs_eurostat.csv", index=False)
flagged[review_cols].to_csv(output_dir / "facilities_flagged_for_review.csv", index=False)

print(f"Summary tables written to {output_dir}")